In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import (
    CarSim,
)
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)
sim = CarSim(prop)


# プログラムの書き方講座2
## 1.プログラムを繰り返すには

例えば、虫を追いかけるプログラムを日本語で書いてみると・・・
```
虫を見つけよ
虫の位置に向かって進め
先頭に戻って繰り返せ
```

### プログラム言語(python)では、Whileというキーワードを使い、段落（左側のスペース）を空けて中にプログラムを書くと繰り返しになる

さきほどのプログラムを、プログラム言語で書くと・・・例えばこうかける。
※   serachとapproachという関数が使える場合

```
While True:
  pos = search(name="bug")
  approach(pos)
```

- ポイント
  - While True: でずっと（無限に）繰り返す。最後の:も忘れないように。
  - 繰り返したい中身は、スペース（2マスや4マス）を使って右に寄せる
  - 繰り返したい中身の、書き出しの位置はそろえる ※左側のスペースの数は同じ

## 2. 変数って何？

- 「〇〇を見つけろ」のような命令は、見つけた結果（値）を返してくれる。
  - 値を記録しておいて、後で使えるようにする為の箱のようなものを「変数」と呼ぶ
  - 箱には好きな名前をつけられる（例えばposのように）

- 下記であれば、虫を見つけた結果をposという名前で記録しておき後で利用できるようにする、という命令になる
```
pos = search(name="bug")
```

記録した値に何が入っているか？どう使うか？は関数によって異なるよ

## 3. 条件に応じて動作を変えよう

- プログラムは、特定の条件が成立するかしないかによって、実行する命令を変えることができる
  - これを「条件分岐」と呼ぶ 
- 例えば「虫がいる方向を向くよう回転する」というプログラムを日本語で書くと
```
虫を見つけよ
・もし虫が左側にいるなら
　・左に回転せよ
・もし虫が右側にいるなら
　・右に回転せよ
・そうでないなら
　・止まれ
先頭に戻って繰り返せ
```

### プログラム言語(python)では、if, elif, else というキーワードを使い、段落（左側のスペース）を空けて中にプログラムを書くと条件分岐になる

例えば、pos.thetaに「見つけた虫の方向を示す角度（0で正面、正なら左側、負なら右側）」の値が入る場合・・・
```
While True:
  pos = search(name="bug")
  if pos.theta > 5:
    rotate(10)
  elif pos.theta < -5:
    rotate(-10)
  else:
    stop()    
```

- ポイント
    - 実行したい命令は、`if 条件文:`もしくは`elif 条件文:`もしくは`else:`の後にスペース（2マスや4マス）を使って右に寄せて書く
    - `if 条件文:` で、「もし、条件文が正しいなら」という意味になる。最後の:も忘れないように。
        - シンプルな条件文は、「値　比較記号　値」の形で書くよ。
            - 比較記号には >, <, >=, <=, ==, !=, is　等が使えるよ
            - 値には「変数」「数値」「文字列」「None」等が使えるよ
        - 条件の例１：　pos.theta > 10
            - 「見つけたモノの方向を示す角度が10度より大きい場合」という条件になる
        - 条件の例２：　10 < pos.theta
            - 例１と同じ意味
        - 条件の例３：　pos.theta == 0 
            - 「見つけたモノの方向を示す角度が0度の場合」という条件になる
            - ※注：ただし、ぴったり0度になることは現実にはほとんど発生しないよ！！！
        - 条件の例４：　pos.name == "bug" 
            - 「見つけたモノの名前がbugである場合」という条件になる
        - 条件の例５：　pos is None 
            - 「見つけたモノが何もない場合」という条件になる
    - 2つ目以降の分岐条件を書く場合は、`elif 条件文:`と書く
    - どの条件も成り立たなかった場合に実行したい命令がある場合は、`else:`と書く
- 補足
    - もっと複雑な条件「〇〇かつ△△」や「〇〇もしくは△△」を書きたい場合は講師に相談してね

- 分岐は多段階にも書けるよ

例えば、「見つけた虫の方向を示す角度」がほぼ正面（5度以下でかつ-5度以上）の時に、前に進む
```
While True:
  pos = search(name="bug")
  if pos.theta <= 5:
    if pos.theta >= -5:
      move()
```

# チュートリアル2

下記の命令を組み合わせてプログラムを書き、ロボットを標識(標識名はsign1)のある場所に向かわせ、3つ目の標識の手前(goal1)で停止しよう。

## 取り組み方
1. 使える命令を理解する
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置やチェックポイント（goal）を把握する
3. ２つ下のセル内にプログラムを書き実行して結果を見る

|使える命令|意味|指定できる値|使い方|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/s]|move(v=0.2)|
|rotate|一定速度で回転する|w=回転速度[度/s]|rotate(w=90)|
|serach|標識を見つける（複数見つかった場合は、最も近いもの）||pos = Search()|
|serach|特定の標識を見つける（複数見つかった場合は、最も近いもの）|name = "標識名"|pos = Search(name="sign1")|

- pos = search() が返す値には下記が含まれる
  - pos.x: 見つけた標識の前方位置[m]　※前方が正
  - pox.y: 見つけた標識の左右位置[m]　※左側が正、右側は負
  - pos.r: 見つけた標識への距離[m]
  - pos.theta: 見つけた標識の角度[度]　※左側が正、右側は負
  - pos.name: 見つけた標識の標識名

## 注意点
- スタート時の位置はランダムに最大20cmほどずれる
- スタート時の向きはランダムに最大1度ほどずれる

In [ ]:
class Tutorial2(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((2.4, 1.5), 0.2, should_stop=True),
        ]
        self.initial_xy = (0.0, 2.0)
        self.random_d_xy = (0.1, 0.2)
        self.random_d_yaw_deg=1
        self.set_signs(
            [
                Sign(x=1.0, y=2.0, name="sign1"),
                Sign(x=1.8, y=1.8, name="sign1"),
                Sign(x=2.6, y=1.4, name="sign1"),
            ]
        )


MissionDrawer(Tutorial2()).show()
print("最大速度", prop.max_velocity, "m/s")
print("最大回転速度", prop.max_rotate_deg, "度/s")

In [ ]:
class Tutorial2(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((2.4, 1.5), 0.2, should_stop=True),
        ]
        self.initial_xy = (0.0, 2.0)
        self.random_d_xy = (0.1, 0.2)
        self.random_d_yaw_deg = 1
        self.set_signs(
            [
                Sign(x=1.0, y=2.0, name="sign1"),
                Sign(x=1.8, y=1.8, name="sign1"),
                Sign(x=2.6, y=1.4, name="sign1"),
            ]
        )

    @staticmethod
    def command_func(*,move, rotate, search, **kwargs):

        while True:
            pos = search()
            if pos is None:
                move(v=0)
            else:
                ######## ここから下にプログラムを書こう
                move(v=1)
                ######## ここより上にプログラムを書こう
        ######## プログラムを書いた後にセルを実行し結果を確認しよう


sim.set_mission(Tutorial2())
sim.run()
SimDrawer(sim).show()